In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [2]:
from datetime import datetime, timedelta
from langchain_core.documents import Document

In [3]:
documents = [
    Document(
        page_content="Subscription pricing is preferred by customers.",
        metadata={
            "topic": "product",
            "source_type": "notes",
            "priority": 2,
            "timestamp": (datetime.now() - timedelta(days=1)).isoformat()
        }
    ),
    Document(
        page_content="Revenue increased by 30% after switching to tiered pricing.",
        metadata={
            "topic": "finance",
            "source_type": "report",
            "priority": 3,
            "timestamp": (datetime.now() - timedelta(days=10)).isoformat()
        }
    ),
    Document(
        page_content="Freemium model failed in enterprise segment.",
        metadata={
            "topic": "product",
            "source_type": "experiment",
            "priority": 5,
            "timestamp": (datetime.now() - timedelta(days=2)).isoformat()
        }
    )
]

In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small",
                                openai_api_key=os.getenv("OPENAI_API_KEY"))

vector_store = Chroma.from_documents(
    documents,
    embedding=embeddings,
    collection_name="router_demo"
)

/Users/home/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [5]:
def retrieve_candidates(query):
    return vector_store.similarity_search_with_score(query, k=5)

In [6]:
from datetime import datetime

def compute_score(doc, similarity_score):

    # Normalize similarity (lower is better in Chroma → convert)
    similarity = 1 / (1 + similarity_score)

    # Priority (higher = more important)
    priority = doc.metadata.get("priority", 1)

    # Recency (more recent = higher score)
    timestamp_str = doc.metadata.get("timestamp")
    if timestamp_str:
        timestamp = datetime.fromisoformat(timestamp_str)
    else:
        timestamp = datetime.now()
    age_days = (datetime.now() - timestamp).days
    recency = 1 / (1 + age_days)

    # Final weighted score
    final_score = (
        0.5 * similarity +
        0.3 * priority +
        0.2 * recency
    )

    return final_score

In [7]:
def rank_contexts(results):

    scored = []

    for doc, sim_score in results:
        score = compute_score(doc, sim_score)
        scored.append((doc, score))

    # Sort descending
    scored.sort(key=lambda x: x[1], reverse=True)

    return scored

In [8]:
def resolve_conflicts(scored_docs):

    selected = []
    seen_topics = set()

    for doc, score in scored_docs:

        topic = doc.metadata.get("topic")

        # If already have context for this topic, skip weaker ones
        if topic in seen_topics:
            continue

        selected.append(doc)
        seen_topics.add(topic)

    return selected

In [9]:
def context_router(query):

    # Step 1: Retrieve
    results = retrieve_candidates(query)

    # Step 2: Rank
    ranked = rank_contexts(results)

    # Step 3: Resolve conflicts
    final_context = resolve_conflicts(ranked)

    return final_context

In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o", temperature=0)

def generate_answer(query):

    contexts = context_router(query)

    combined_context = "\n\n".join(
        [doc.page_content for doc in contexts]
    )

    messages = [
        SystemMessage(content="You are a product strategist."),
        HumanMessage(content=f"""
Context:
{combined_context}

Question:
{query}
""")
    ]

    response = llm.invoke(messages)

    return response.content

In [11]:
query = "What pricing strategy should we adopt?"

answer = generate_answer(query)

print(answer)

Given the context that the freemium model failed in the enterprise segment and revenue increased by 30% after switching to tiered pricing, it seems that a tiered pricing strategy is more effective for your business in this segment. Here are some considerations and recommendations for adopting a pricing strategy:

1. **Refine Tiered Pricing Structure**:
   - **Define Clear Value Propositions**: Ensure each tier offers distinct and compelling value propositions that align with the needs of different customer segments.
   - **Segment Based on Usage and Features**: Differentiate tiers based on usage limits, feature sets, or levels of customer support. This allows customers to choose a plan that best fits their needs and budget.
   - **Enterprise-Specific Tiers**: Consider creating a dedicated enterprise tier with advanced features, customization options, and premium support to cater specifically to larger organizations.

2. **Value-Based Pricing**:
   - **Understand Customer Needs**: Condu